In [15]:
import sympy as sp
from sympy.physics.vector import dynamicsymbols
from sympy.physics.mechanics import init_vprinting

# Set dot notation for time derivatives
init_vprinting()
theta, x, F = dynamicsymbols('theta x F')
m, M, r, g, b, c = sp.symbols('m M r g b c')

T = 0.5 * x.diff() ** 2 * (m + M) + m * r * sp.cos(theta) * theta.diff() * x.diff() + 0.5 * m * r ** 2 * theta.diff() ** 2
V = m * g * r * sp.cos(theta)
L = T - V
L

In [16]:
# Euler-Lagrange equation for theta
dL_dtheta = L.diff(theta)
dL_dtheta_diff = L.diff(theta.diff())
d_dt_dL_dtheta_diff = dL_dtheta_diff.diff(sp.symbols('t'))
eq_theta = sp.Eq(d_dt_dL_dtheta_diff - dL_dtheta, -c * theta.diff())
eq_theta

In [17]:
# Euler-Lagrange equation for x
dL_dx = L.diff(x)
dL_dx_diff = L.diff(x.diff())
d_dt_dL_dx_diff = dL_dx_diff.diff(sp.symbols('t'))
eq_x = sp.Eq(d_dt_dL_dx_diff - dL_dx, F - b * x.diff())
eq_x

In [18]:
# To matrix form
eq_x_lin = eq_x.subs({sp.sin(theta): theta, sp.cos(theta): 1, theta.diff() ** 2: 0})
eq_theta_lin = eq_theta.subs({sp.sin(theta): theta, sp.cos(theta): 1, theta.diff() ** 2: 0})

eq_x_alg = eq_x_lin.lhs - eq_x_lin.rhs
eq_theta_alg = eq_theta_lin.lhs - eq_theta_lin.rhs
eq_theta_alg

In [19]:
# 4x4 linearized state-space model
xddot, thetaddot = sp.symbols('xddot thetaddot')
xdot, thetadot = sp.symbols('xdot thetadot')
u = sp.symbols('u')

eq_x_alg = eq_x_alg.subs({x.diff().diff(): xddot, x.diff(): xdot, theta.diff().diff(): thetaddot, theta.diff(): thetadot, F: u})
eq_theta_alg = eq_theta_alg.subs({x.diff().diff(): xddot, x.diff(): xdot, theta.diff().diff(): thetaddot, theta.diff(): thetadot, F: u})

sol_lin = sp.solve([eq_x_alg, eq_theta_alg], [xddot, thetaddot], dict=True)[0]

state = sp.Matrix([x, xdot, theta, thetadot])
f = sp.Matrix([xdot, sol_lin[xddot], thetadot, sol_lin[thetaddot]])

A_state = sp.simplify(f.jacobian(state))
B_state = sp.simplify(f.jacobian(sp.Matrix([u])))

A_state, B_state

⎛⎡0   1       0          0     ⎤       ⎞
⎜⎢                             ⎥  ⎡ 0 ⎤⎟
⎜⎢   -b     -g⋅m         c     ⎥  ⎢   ⎥⎟
⎜⎢0  ───    ─────       ───    ⎥  ⎢ 1 ⎥⎟
⎜⎢    M       M         M⋅r    ⎥  ⎢ ─ ⎥⎟
⎜⎢                             ⎥  ⎢ M ⎥⎟
⎜⎢0   0       0          1     ⎥, ⎢   ⎥⎟
⎜⎢                             ⎥  ⎢ 0 ⎥⎟
⎜⎢    b   g⋅(M + m)  c⋅(-M - m)⎥  ⎢   ⎥⎟
⎜⎢0  ───  ─────────  ──────────⎥  ⎢-1 ⎥⎟
⎜⎢   M⋅r     M⋅r            2  ⎥  ⎢───⎥⎟
⎝⎣                     M⋅m⋅r   ⎦  ⎣M⋅r⎦⎠

In [20]:
# Nonlinear state-space model
xddot, thetaddot = sp.symbols('xddot thetaddot')
eq_x_alg = eq_x.subs({theta.diff(): thetadot, theta.diff().diff(): thetaddot, x.diff(): xdot, x.diff().diff(): xddot, F: u})
eq_theta_alg = eq_theta.subs({theta.diff(): thetadot, theta.diff().diff(): thetaddot, x.diff(): xdot, x.diff().diff(): xddot, F: u})

sol_lin = sp.solve([eq_x_alg, eq_theta_alg], [xddot, thetaddot], dict=True)[0]

state = sp.Matrix([theta, thetadot, x, xdot])
f = sp.Matrix([thetadot, sol_lin[thetaddot], xdot, sol_lin[xddot]])

A_state_nl = sp.simplify(f.jacobian(state))
B_state_nl = sp.simplify(f.jacobian(sp.Matrix([u])))

A_state_nl, B_state_nl

⎛⎡                                                                             ↪
⎜⎢                                                                             ↪
⎜⎢                                                                             ↪
⎜⎢  ⎛         2   ⎞ ⎛                                              2    2      ↪
⎜⎢r⋅⎝M + m⋅sin (θ)⎠⋅⎝M⋅g⋅cos(θ) - b⋅ẋ⋅sin(θ) + g⋅m⋅cos(θ) - 2⋅m⋅r⋅θ̇ ⋅cos (θ) + ↪
⎜⎢                                                                             ↪
⎜⎢──────────────────────────────────────────────────────────────────────────── ↪
⎜⎢                                                                             ↪
⎜⎢                                                                             ↪
⎜⎢                                                                             ↪
⎜⎢                                                                             ↪
⎜⎢                                                                             ↪
⎜⎢                        

In [21]:
# print A and B in numpy format
import numpy as np

A_state_np = sp.lambdify((m, M, r, g, b, c), A_state, 'numpy')
B_state_np = sp.lambdify((m, M, r, g, b, c), B_state, 'numpy')

params = (1.0, 5.0, 0.5, 9.81, 0.1, 0.1)

A_num = np.array(A_state_np(*params), dtype=float)
B_num = np.array(B_state_np(*params), dtype=float)

print('A_state = np.array(')
print(np.array2string(A_num, separator=', '))
print(')')

print('B_state = np.array(')
print(np.array2string(B_num, separator=', '))
print(')')

A_state = np.array(
[[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00],
 [ 0.0000e+00, -2.0000e-02, -1.9620e+00,  4.0000e-02],
 [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00],
 [ 0.0000e+00,  4.0000e-02,  2.3544e+01, -4.8000e-01]]
)
B_state = np.array(
[[ 0. ],
 [ 0.2],
 [ 0. ],
 [-0.4]]
)
